# Irodori-TTS Unified Playground：Colab T4 推論與前後端測試

這份 notebook 會在 Google Colab 的 T4 GPU 上啟動完整的 Irodori-TTS Unified Playground：FastAPI 後端、React/Vite 前端，以及一個最小 VoiceDesign / no-ref 推論測試。

**預期時間**：第一次安裝與下載模型約 20-45 分鐘，實際取決於 Colab、PyTorch/uv resolver、Hugging Face 下載速度。

**預估 VRAM**：T4 約 15GB VRAM。這份 notebook 預設不使用 bf16，因為 T4 不支援 bf16；如果拿到 A100/L4，後端會自動選擇較適合的 precision。

**會做什麼**：
- clone `linmino/Irodori-tts-ui-playground-zh`
- clone upstream `Aratako/Irodori-TTS`
- 使用 upstream 官方建議的 `uv sync` 建立推論 venv
- 用該 venv 啟動本專案 FastAPI 後端
- 啟動 Vite 前端並用 Colab port viewer 打開
- 透過 API 載入 VoiceDesign 模型並生成一段測試音訊


## 步驟 0：前置需求

請先在 Colab 選單確認：**執行階段 → 變更執行階段類型 → T4 GPU**。

這份 notebook 不需要 Hugging Face token，因為目前使用的公開模型是公開下載；如果未來模型卡改為 gated，請在 Colab Secrets 或環境變數設定 `HF_TOKEN`。

**如果壞了**：最常見是沒有拿到 GPU、Colab 分配到非 T4、或下載模型時網路中斷。後面有 log 與重置 cell 可以協助排查。

In [ ]:
# Cell 1：確認 Colab runtime 與 GPU
import subprocess
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "未偵測到 GPU。請到『執行階段 → 變更執行階段類型 → T4 GPU』後重試。"

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('GPU:', gpu_name)
print('VRAM GB:', round(total_vram_gb, 2))
print('CUDA capability:', capability, '(T4 通常是 (7, 5))')
print('bf16 supported:', torch.cuda.is_bf16_supported())
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.free,driver_version', '--format=csv']).decode())

if 'T4' not in gpu_name:
    print('⚠️ 你目前拿到的不是 T4。這份 notebook 仍可試跑，但預估 VRAM/速度可能不同。')

## 步驟 1：安裝基礎工具與設定路徑

Irodori-TTS upstream 使用 `uv sync` 管理環境。這裡讓 upstream 建立自己的 `.venv`，避免覆蓋 Colab kernel 內建的 Python 套件，也比較不容易讓 notebook runtime 壞掉。

**為什麼不直接在 Colab 全域 pip install 全部套件**：Irodori-TTS 對 PyTorch / torchcodec / DACVAE 有較新的需求，直接污染 kernel 容易讓後續 cell 或 Colab 預裝套件衝突。用 venv 啟動後端服務會穩很多。

In [ ]:
# Cell 2：安裝基礎工具、建立路徑變數
from pathlib import Path
import os

PROJECT_REPO = 'https://github.com/linmino/Irodori-tts-ui-playground-zh.git'
UPSTREAM_REPO = 'https://github.com/Aratako/Irodori-TTS.git'

PROJECT_DIR = Path('/content/Irodori-tts-ui-playground-zh')
UPSTREAM_DIR = Path('/content/Irodori-TTS')
VENV_PYTHON = UPSTREAM_DIR / '.venv' / 'bin' / 'python'

BACKEND_PORT = 8000
FRONTEND_PORT = 5173
LOG_DIR = Path('/content/irodori_logs')
OUTPUT_ARCHIVE_DIR = Path('/content/irodori_saved_outputs')

LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

# Colab 上 ffmpeg/libsndfile 對音訊讀寫很重要；uv 固定版本，避免未來行為漂移太大。
!apt-get update -qq
!apt-get install -y -qq git ffmpeg libsndfile1 curl lsof
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
!python -m pip install -q uv==0.8.15 requests==2.32.4 packaging==24.2

os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('PROJECT_DIR =', PROJECT_DIR)
print('UPSTREAM_DIR =', UPSTREAM_DIR)


## 步驟 2：下載本專案與 upstream Irodori-TTS

本專案只包含前後端 wrapper；真正的推論核心來自 upstream `Aratako/Irodori-TTS`。我們分開 clone，之後用 upstream 的 venv 來啟動本專案後端。

In [ ]:
# Cell 3：clone / 更新 repo
import subprocess

def clone_or_pull(repo_url: str, target: Path):
    if target.exists():
        print(f'更新 {target} ...')
        subprocess.run(['git', 'pull', '--ff-only'], cwd=target, check=True)
    else:
        print(f'clone {repo_url} -> {target}')
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(target)], check=True)

clone_or_pull(PROJECT_REPO, PROJECT_DIR)
clone_or_pull(UPSTREAM_REPO, UPSTREAM_DIR)

print('✅ repo 準備完成')

## 步驟 3：建立 Irodori-TTS 推論 venv 與前端依賴

這一步最久，因為 `uv sync` 會依照 upstream pyproject 安裝推論需要的 PyTorch / torchcodec / dacvae / transformers 等套件。

**T4 注意**：T4 不支援 bf16，所以後端 `auto` precision 已改成「GPU 支援 bf16 才用 bf16，否則使用 fp32」。500M 推論在 T4 fp32 會比高階 GPU 慢，但能避開 bf16 crash。

**如果壞了**：先看錯誤是不是網路下載中斷。通常重跑本 cell 可以續裝；如果是 CUDA/PyTorch wheel 問題，請先確認 Colab runtime 是 GPU。

In [ ]:
# Cell 4：安裝 upstream venv、本專案後端依賴、前端 npm 套件
import subprocess

# upstream 官方建議：uv sync。這會在 /content/Irodori-TTS/.venv 建立推論環境。
subprocess.run(['uv', 'sync', '--no-dev'], cwd=UPSTREAM_DIR, check=True)

# 讓 upstream 原始碼固定出現在 venv import path；這比只靠 PYTHONPATH 更不容易被背景服務環境影響。
site_packages = Path(subprocess.check_output([
    str(VENV_PYTHON), '-c', 'import site; print(site.getsitepackages()[0])'
], text=True).strip())
(site_packages / 'irodori_tts_upstream.pth').write_text(str(UPSTREAM_DIR) + '\n', encoding='utf-8')
print('venv site-packages =', site_packages)

# 本專案後端用 FastAPI；裝到 upstream venv，讓同一個 Python 同時 import irodori_tts 與 app.main。
backend_packages = [
    'fastapi==0.136.1',
    'uvicorn[standard]==0.46.0',
    'pydantic==2.13.3',
    'python-multipart==0.0.27',
    'huggingface-hub==0.36.0',
    'requests==2.32.4',
]
# uv 建出的 venv 不一定包含 pip；用 uv pip install 直接指定 venv Python，比 python -m pip 穩定。
backend_install_cmd = ['uv', 'pip', 'install', '--python', str(VENV_PYTHON), *backend_packages]
backend_install_result = subprocess.run(backend_install_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if backend_install_result.returncode != 0:
    print('❌ backend package install failed. uv output tail:')
    print(backend_install_result.stdout[-6000:])
    backend_install_result.check_returncode()

# 前端使用 Vite；package-lock 已鎖版本，npm install 會依 lockfile 安裝。
subprocess.run(['npm', 'install'], cwd=PROJECT_DIR / 'frontend', check=True)

print('✅ Python venv 與前端依賴安裝完成')

In [ ]:
# Cell 5：確認 venv 能 import 必要套件
import os

check_code = r'''
import importlib
import traceback

def check_import(module_name):
    print(f'checking import {module_name} ...', flush=True)
    try:
        module = importlib.import_module(module_name)
    except Exception:
        print(f'❌ import failed: {module_name}', flush=True)
        traceback.print_exc()
        raise
    print(f'✅ {module_name}: {getattr(module, "__version__", "ok")}', flush=True)
    return module

torch = check_import('torch')
print('CUDA available:', torch.cuda.is_available(), flush=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none', flush=True)
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False, flush=True)
check_import('fastapi')
check_import('irodori_tts.inference_runtime')
print('✅ venv import check passed', flush=True)
'''
check_env = os.environ.copy()
for key in ['MPLBACKEND', 'PYTHONPATH', 'JPY_PARENT_PID', 'JPY_SESSION_NAME', 'CLICOLOR', 'FORCE_COLOR']:
    check_env.pop(key, None)
check_env.update({
    'MPLBACKEND': 'Agg',
    'PYTHONUNBUFFERED': '1',
    'PYTHONPATH': str(UPSTREAM_DIR),
    'HF_HOME': '/content/hf_cache',
    'TOKENIZERS_PARALLELISM': 'false',
})

check_result = subprocess.run(
    [str(VENV_PYTHON), '-c', check_code],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=check_env,
)
print(check_result.stdout[-8000:])
check_result.check_returncode()

## 步驟 4：服務管理工具

Colab 啟動背景服務不要用 `!command &`，因為 Jupyter 會把一些只適合 notebook 的環境變數傳進子 process，常見結果是 `MPLBACKEND` 或 `PYTHONPATH` 污染。下面用 `subprocess.Popen`、乾淨環境、log 檔與 port wait 來啟動服務。

In [ ]:
# Cell 6：背景服務 helper
import os
import socket
import time
import subprocess
from pathlib import Path

JUPYTER_INJECTED = (
    'MPLBACKEND', 'PYTHONPATH', 'JPY_PARENT_PID', 'JPY_SESSION_NAME',
    'CLICOLOR', 'FORCE_COLOR', 'PYDEVD_DISABLE_FILE_VALIDATION',
)

def make_clean_env(extra=None):
    env = os.environ.copy()
    for key in JUPYTER_INJECTED:
        env.pop(key, None)
    env.update({
        'MPLBACKEND': 'Agg',
        'PYTHONUNBUFFERED': '1',
        'PYTHONPATH': str(UPSTREAM_DIR),
        'HF_HOME': '/content/hf_cache',
        'TOKENIZERS_PARALLELISM': 'false',
    })
    if extra:
        env.update(extra)
    return env

def kill_port(port: int):
    subprocess.run(['fuser', '-k', f'{port}/tcp'], capture_output=True)
    time.sleep(1)

def wait_for_port(port: int, proc, log_path: Path, idle_secs=420):
    start = time.time()
    last_size = 0
    last_change = time.time()
    while True:
        if proc.poll() is not None:
            tail = ''
            if log_path.exists():
                tail = ''.join(log_path.read_text(errors='replace').splitlines(True)[-80:])
            raise RuntimeError(f'process exit code={proc.returncode}\n最後 log:\n{tail}')
        with socket.socket() as s:
            s.settimeout(0.5)
            try:
                s.connect(('127.0.0.1', port))
                print(f'✅ port {port} 已就緒，耗時 {time.time() - start:.0f}s')
                return
            except (ConnectionRefusedError, socket.timeout):
                pass
        if log_path.exists():
            cur_size = log_path.stat().st_size
            if cur_size > last_size:
                with log_path.open(errors='replace') as f:
                    f.seek(last_size)
                    new = f.read()
                if new.strip():
                    for line in new.splitlines()[-12:]:
                        print('  ▸', line)
                last_size = cur_size
                last_change = time.time()
        if time.time() - last_change > idle_secs:
            raise TimeoutError(f'連續 {idle_secs}s 沒有 log 進度，請檢查 {log_path}')
        time.sleep(2)

def tail_log(log_path: Path, lines=80):
    if not log_path.exists():
        print('log 不存在:', log_path)
        return
    print(''.join(log_path.read_text(errors='replace').splitlines(True)[-lines:]))

print('✅ helper ready')

## 步驟 5：啟動 FastAPI 後端

後端會先啟動 metadata API；真正吃 VRAM 的模型會在你點 UI 的「載入模型」或後面 API 測試 cell 時才下載與載入。

In [ ]:
# Cell 7：啟動後端 API
import requests

BACKEND_LOG = LOG_DIR / 'backend.log'
kill_port(BACKEND_PORT)

backend_env = make_clean_env()
preflight_code = 'from irodori_tts.inference_runtime import InferenceRuntime, RuntimeKey; print("backend irodori_tts import ok")'
preflight_result = subprocess.run(
    [str(VENV_PYTHON), '-c', preflight_code],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=backend_env,
)
print(preflight_result.stdout[-4000:])
preflight_result.check_returncode()

backend_fh = BACKEND_LOG.open('w')
BACKEND_PROC = subprocess.Popen(
    [
        str(VENV_PYTHON), '-m', 'uvicorn', 'app.main:app',
        '--host', '0.0.0.0', '--port', str(BACKEND_PORT),
        '--app-dir', str(PROJECT_DIR / 'backend'),
    ],
    cwd=PROJECT_DIR,
    env=backend_env,
    stdout=backend_fh,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print('backend PID =', BACKEND_PROC.pid, 'log =', BACKEND_LOG)
wait_for_port(BACKEND_PORT, BACKEND_PROC, BACKEND_LOG)

health = requests.get(f'http://127.0.0.1:{BACKEND_PORT}/api/v1/health', timeout=10).json()
print(health)

## 步驟 6：啟動 Vite 前端

前端使用 Vite dev server，會把 `/api` proxy 到 `127.0.0.1:8000`。因此瀏覽器只需要打開前端 port，API 會由 Vite 在 Colab 內部轉發，不需要另外處理 CORS。

In [ ]:
# Cell 8：啟動前端
FRONTEND_LOG = LOG_DIR / 'frontend.log'
kill_port(FRONTEND_PORT)

COLAB_VITE_CONFIG = PROJECT_DIR / 'frontend' / 'vite.config.colab.ts'
COLAB_VITE_CONFIG.write_text(
    '''import { defineConfig } from "vite";\nimport react from "@vitejs/plugin-react";\n\nexport default defineConfig({\n  plugins: [react()],\n  server: {\n    host: "0.0.0.0",\n    port: 5173,\n    allowedHosts: true,\n    proxy: {\n      "/api": "http://127.0.0.1:8000"\n    }\n  }\n});\n''',
    encoding='utf-8',
)

frontend_fh = FRONTEND_LOG.open('w')
FRONTEND_PROC = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--config', str(COLAB_VITE_CONFIG), '--host', '0.0.0.0', '--port', str(FRONTEND_PORT)],
    cwd=PROJECT_DIR / 'frontend',
    env=make_clean_env(),
    stdout=frontend_fh,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print('frontend PID =', FRONTEND_PROC.pid, 'log =', FRONTEND_LOG)
wait_for_port(FRONTEND_PORT, FRONTEND_PROC, FRONTEND_LOG, idle_secs=180)
print('✅ 前端已啟動')

## 步驟 7：用 Cloudflare Tunnel 打開前端

這裡會用 `cloudflared tunnel --url http://127.0.0.1:5173` 建立臨時公開網址。打開後你應該看到 Unified Playground。先確認右上方狀態列與 tabs 正常，再試著在 UI 載入 VoiceDesign 模型。

**為什麼用 Cloudflare Tunnel**：Colab port viewer 有時會受 notebook UI 限制；Cloudflare Tunnel 會給一個可直接貼到瀏覽器的新 URL，更適合測試前端與後端 proxy。

**如果畫面空白**：先 hard refresh；如果還是不行，執行後面的 log cell 看 `frontend.log`、`backend.log` 與 `cloudflared.log`。

In [ ]:
# Cell 9：啟動 Cloudflare Tunnel 並列印公開網址
import re
import socket
from urllib.parse import urlparse

import requests

TUNNEL_LOG = LOG_DIR / 'cloudflared.log'
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)

tunnel_fh = TUNNEL_LOG.open('w')
TUNNEL_PROC = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{FRONTEND_PORT}', '--no-autoupdate'],
    cwd=PROJECT_DIR,
    env=make_clean_env(),
    stdout=tunnel_fh,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print('cloudflared PID =', TUNNEL_PROC.pid, 'log =', TUNNEL_LOG)

public_url = None
start = time.time()
while time.time() - start < 90:
    if TUNNEL_PROC.poll() is not None:
        tail_log(TUNNEL_LOG, lines=80)
        raise RuntimeError(f'cloudflared exited: {TUNNEL_PROC.returncode}')
    if TUNNEL_LOG.exists():
        text = TUNNEL_LOG.read_text(errors='replace')
        match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', text)
        if match:
            public_url = match.group(0)
            break
    time.sleep(2)

if not public_url:
    tail_log(TUNNEL_LOG, lines=120)
    raise TimeoutError('90 秒內沒有取得 trycloudflare URL。請重跑本 cell 或檢查 cloudflared.log。')

print('✅ Cloudflare Tunnel URL:')
print(public_url)

try:
    from google.colab import output as colab_output
    colab_proxy_url = colab_output.eval_js(f'google.colab.kernel.proxyPort({FRONTEND_PORT})')
    print('\nColab proxy fallback URL:')
    print(colab_proxy_url)
except Exception as exc:
    print('Colab proxy URL unavailable:', repr(exc))

def wait_for_public_url(url: str, timeout_secs: int = 180):
    host = urlparse(url).hostname
    deadline = time.time() + timeout_secs
    last_error = None
    attempt = 0
    print(f'等待 trycloudflare DNS/HTTP 就緒，最多 {timeout_secs} 秒 ...')
    while time.time() < deadline:
        attempt += 1
        if TUNNEL_PROC.poll() is not None:
            tail_log(TUNNEL_LOG, lines=80)
            raise RuntimeError(f'cloudflared exited while waiting for public URL: {TUNNEL_PROC.returncode}')
        try:
            socket.gethostbyname(host)
            smoke = requests.get(url, timeout=15)
            print('public URL status:', smoke.status_code)
            if smoke.status_code < 400:
                return smoke
            print('response head:')
            print(smoke.text[:1200])
            smoke.raise_for_status()
        except Exception as exc:
            last_error = exc
            if attempt == 1 or attempt % 6 == 0:
                print(f'  still waiting: {type(exc).__name__}: {exc}')
            time.sleep(5)
    print('\nfrontend.log tail:')
    tail_log(FRONTEND_LOG, lines=80)
    print('\ncloudflared.log tail:')
    tail_log(TUNNEL_LOG, lines=80)
    raise TimeoutError(f'public URL 仍未就緒，最後錯誤：{last_error!r}。可重跑本 cell 取得新的 quick tunnel，或先使用 Colab proxy fallback URL。')

wait_for_public_url(public_url)

print('\n打開這個網址即可使用前端。Vite 會把 /api proxy 到 Colab 內部後端。')

## 步驟 8：API 推論 smoke test（VoiceDesign，建議先跑）

這個 cell 會直接打後端 API：先載入 VoiceDesign 模型，再用一段很短的文字生成音訊。第一次載入會下載 Hugging Face 權重與 codec，會比較久。

**為什麼先測 VoiceDesign**：它不需要 reference audio，變因最少，適合確認模型下載、GPU 載入、後端 job、WAV 輸出都能通。

**如果 OOM**：執行後面的 OOM/卸載 cell，重跑時把 `num_steps` 降到 12-20，並確認只載入一個模型。

In [ ]:
# Cell 10：VoiceDesign API 推論測試
import time
import requests
from IPython.display import Audio, display

base_url = f'http://127.0.0.1:{BACKEND_PORT}/api/v1'
print('載入前 VRAM:')
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv

load_payload = {
    'mode': 'voicedesign',
    'checkpoint': 'Aratako/Irodori-TTS-500M-v2-VoiceDesign',
    'model_device': 'auto',
    'model_precision': 'auto',   # T4 會落到 fp32，避免 bf16 crash
    'codec_device': 'auto',
    'codec_precision': 'auto',
}
r = requests.post(f'{base_url}/models/load', json=load_payload, timeout=900)
print('load status:', r.status_code)
print(r.text[:1000])
r.raise_for_status()

payload = {
    'mode': 'voicedesign',
    'text': 'こんにちは、今日はとてもいい天気ですね。',
    'caption': '落ち着いた女性の声で、近い距離感でやわらかく自然に読み上げてください。',
    'no_ref': True,
    'num_steps': 24,             # T4 smoke test 建議 20-40；越高越慢
    'num_candidates': 1,         # T4 先用 1，避免一次生成多個吃 VRAM
    'seed': 1234,
    'cfg_guidance_mode': 'independent',
    'cfg_scale_text': 3.0,
    'cfg_scale_caption': 3.0,
    'cfg_min_t': 0.5,
    'cfg_max_t': 1.0,
    'context_kv_cache': True,
}
job = requests.post(f'{base_url}/generations', json=payload, timeout=60).json()
print('job:', job)

while True:
    meta = requests.get(f'{base_url}/generations/{job["id"]}', timeout=30).json()
    print('status:', meta['status'])
    if meta['status'] == 'completed':
        break
    if meta['status'] == 'failed':
        raise RuntimeError(meta.get('error'))
    time.sleep(2)

audio_url = f'http://127.0.0.1:{BACKEND_PORT}{meta["audios"][0]["url"]}'
wav_path = OUTPUT_ARCHIVE_DIR / f'{meta["id"]}_voice_design.wav'
wav_path.write_bytes(requests.get(audio_url, timeout=60).content)
print('WAV saved:', wav_path)
print('metadata:', {k: meta.get(k) for k in ['id', 'elapsed_seconds', 'duration_seconds', 'rtf', 'seed_used']})
display(Audio(str(wav_path)))

print('生成後 VRAM:')
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv

## 步驟 9：可選 no-ref Base 模型測試

這會切換到 base/reference 模型。後端會先卸載 VoiceDesign，再載入 Base；這符合本專案「同一時間只允許一個模型」的 VRAM 策略。

第一次載入 Base 會再次下載權重。若只想測前端與 VoiceDesign，這格可以跳過。

In [ ]:
# Cell 11：可選 Base no-ref 推論測試
# 如果 T4 VRAM 壓力大或你只要測 UI，可以先不要跑這格。
load_payload = {
    'mode': 'reference',
    'checkpoint': 'Aratako/Irodori-TTS-500M-v2',
    'model_device': 'auto',
    'model_precision': 'auto',
    'codec_device': 'auto',
    'codec_precision': 'auto',
}
r = requests.post(f'{base_url}/models/load', json=load_payload, timeout=900)
print('load status:', r.status_code)
print(r.text[:1000])
r.raise_for_status()

payload = {
    'mode': 'reference',
    'text': '今日は短いテストです。',
    'no_ref': True,
    'num_steps': 20,
    'num_candidates': 1,
    'seed': 5678,
    'cfg_guidance_mode': 'independent',
    'cfg_scale_text': 3.0,
    'cfg_scale_speaker': 5.0,
    'cfg_min_t': 0.5,
    'cfg_max_t': 1.0,
    'context_kv_cache': True,
}
job = requests.post(f'{base_url}/generations', json=payload, timeout=60).json()
print('job:', job)

while True:
    meta = requests.get(f'{base_url}/generations/{job["id"]}', timeout=30).json()
    print('status:', meta['status'])
    if meta['status'] == 'completed':
        break
    if meta['status'] == 'failed':
        raise RuntimeError(meta.get('error'))
    time.sleep(2)

audio_url = f'http://127.0.0.1:{BACKEND_PORT}{meta["audios"][0]["url"]}'
wav_path = OUTPUT_ARCHIVE_DIR / f'{meta["id"]}_base_no_ref.wav'
wav_path.write_bytes(requests.get(audio_url, timeout=60).content)
print('WAV saved:', wav_path)
display(Audio(str(wav_path)))

## Debug：查看 log

如果 UI 沒反應、模型載入失敗、或 API 回傳 500，先看後端 log。前端畫面問題則看 frontend log。

In [ ]:
# Cell 12：查看最近 log
print('\n===== backend.log =====')
tail_log(BACKEND_LOG, lines=120)
print('\n===== frontend.log =====')
tail_log(FRONTEND_LOG, lines=80)
print('\n===== cloudflared.log =====')
try:
    tail_log(TUNNEL_LOG, lines=80)
except NameError:
    print('Cloudflare Tunnel 尚未啟動。')

## OOM / 重置服務

如果看到 CUDA out of memory，先卸載模型與清 GPU cache，再把 `num_steps` 或 `num_candidates` 降低。若服務狀態混亂，執行重置 cell 後從「啟動後端」開始重跑。

In [ ]:
# Cell 13：卸載模型並清理 GPU cache
import gc
try:
    print(requests.post(f'{base_url}/models/unload', timeout=60).json())
except Exception as exc:
    print('卸載 API 失敗，可能後端沒啟動:', exc)
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv

In [ ]:
# Cell 14：重置所有服務（跑壞時用）
for port in [BACKEND_PORT, FRONTEND_PORT]:
    kill_port(port)
subprocess.run(['pkill', '-f', 'uvicorn app.main:app'], capture_output=True)
subprocess.run(['pkill', '-f', 'vite'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
print('✅ 已嘗試清掉後端/前端服務。請從 Cell 7 重新啟動。')

## 可選：保存 outputs 到 Google Drive

Colab 的 `/content` 會在 runtime 結束後消失。如果你想保留生成音檔與 metadata，執行下面 cell 掛載 Drive 並打包保存。

In [ ]:
# Cell 15：可選保存 outputs 到 Drive
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive')
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_dir = Path('/content/drive/MyDrive/Irodori-TTS-Colab')
drive_dir.mkdir(parents=True, exist_ok=True)
archive_path = drive_dir / f'outputs_{stamp}.tar.gz'
subprocess.run(['tar', '-czf', str(archive_path), '-C', str(PROJECT_DIR), 'outputs'], check=False)
print('已保存到:', archive_path)

## 完成

你現在已經在 Colab T4 上跑起完整前後端，並可透過 UI 或 API 測試 Irodori-TTS 推論。

**產出位置**：
- 後端生成 metadata / WAV：`/content/Irodori-tts-ui-playground-zh/outputs/`
- notebook API smoke test 複製出的 WAV：`/content/irodori_saved_outputs/`
- logs：`/content/irodori_logs/`

**常見問題**：
- `CUDA out of memory`：先跑卸載 cell，降低 `num_steps`、保持 `num_candidates=1`。
- UI 可以開但生成失敗：看 `backend.log`，多半是模型下載、precision 或套件 import 問題。
- 前端空白：看 `frontend.log`，或重跑前端啟動 cell。
- 下載太久：Hugging Face / Colab 網路偶爾不穩，重跑載入模型 cell 通常可續用 cache。

**正式部署提醒**：這份 notebook 是 Colab 測試用途，服務只應在你的 Colab runtime 內短時間使用；公開部署時請重新設計 CORS、認證、檔案清理與資源限制。